# Main Procedure for Data Handling, Modelling, and Evaluating

## Dependancies

In [2]:
from utils import drainage
import pandas as pd
from PIL import Image 
import matplotlib.pyplot as plt
import geopandas as gpd
import numpy as np

In [2]:
full_data = pd.read_excel('./data/20260316_Fish_Data.xlsx', sheet_name='Fish')

## Create Drainage Area to Waterbody Mapping Dataset
Since the NHDES assemblage data does not contain the drainage area of each location, utilize the USGS watershed data based on each unqiue_id of each collection location for watershed data. Save shape file data to a csv with new column for area by sq miles. 

In [ ]:
gdf = gpd.read_file("./src/data/Shapefiles/All_Watersheds_wJoin.zip")

# Project to UTM zone 19N (covers New Hampshire/Vermont area) for accurate area calculation
projected = gdf.to_crs(epsg=32619)

# Calculate area in square miles (1 sq meter = 3.86102e-7 sq miles)
SQ_M_TO_SQ_MI = 3.86102e-7
projected["area (sq mi)"] = projected.geometry.area * SQ_M_TO_SQ_MI

# Copy the new column back to the original GDF (drop geometry for CSV export)
gdf["area (sq mi)"] = projected["area (sq mi)"]

output_df = gdf.drop(columns="geometry")
output_df.to_csv("./src/data/watersheds_with_area_based-on-shp.csv", index=False)

print(f"Saved {len(output_df)} records to watersheds_with_area_based-on-shp.csv")
print(output_df[["Name", "Unique_ID", "area (sq mi)"]].head(10))

# Extract NLCD Landuse Data
We want to determine what extent of human interaction should disqualify collection sites from inclusion. We do this by using NLCD land use data for each year. 

In [11]:
img = Image.open(f"data\\NLCD tiffs\\Annual_NLCD_LndCov_2002_CU_C1V1.tiff")
array = np.array(img)
print(np.unique(array))

[ 11  21  22  23  24  31  41  42  43  52  71  81  82  90  95 250]


In [16]:

# Build species presence/absence CSV
import pandas as pd
import re
import io

fish = pd.read_excel('./data/20260316_Fish_Data.xlsx', sheet_name='Fish')

# Some waterbody names contain commas (e.g. "Gale River, NB, US of Dam") and are
# quoted, but with trailing spaces before the closing delimiter — fix before parsing.
with open('./data/watersheds_with_area_based-on-shp.csv', encoding='utf-8') as f:
    raw = f.read()
fixed = re.sub(r'"([^"]*)"\s+,', r'"\1",', raw)
watersheds = pd.read_csv(io.StringIO(fixed))

# Per (Unique_ID, species): sum total individuals collected
totals = (
    fish.groupby(['Unique_ID', 'Abbreviation'])['Individuals']
    .sum()
    .reset_index()
)

# Compute cw_ind: 1 if sum of EBT + SS individuals at site >= 30, else 0
cw = (
    totals[totals['Abbreviation'].isin(['EBT', 'SS'])]
    .groupby('Unique_ID')['Individuals']
    .sum()
    .reset_index()
)
cw['cw_ind'] = (cw['Individuals'] >= 30).astype(int)
cw = cw[['Unique_ID', 'cw_ind']]

# Pivot: rows = Unique_ID, columns = species abbreviations (excluding EBT and SS)
pivot = totals.pivot_table(index='Unique_ID', columns='Abbreviation', values='Individuals', aggfunc='sum', fill_value=0)
pivot = pivot.drop(columns=['NO FISH', 'EBT', 'SS'], errors='ignore')

# Encode presence (>=30) as 1, absence (<30) as 0
species_cols = pivot.columns.tolist()
pivot[species_cols] = (pivot[species_cols] >= 30).astype(int)
pivot = pivot.reset_index()

# Lat/lon: use first recorded value per Unique_ID from fish data
coords = (
    fish.dropna(subset=['Lat_Dec_USGS', 'Long_Dec_USGS'])
    .groupby('Unique_ID')[['Lat_Dec_USGS', 'Long_Dec_USGS']]
    .first()
    .reset_index()
    .rename(columns={'Lat_Dec_USGS': 'latitude', 'Long_Dec_USGS': 'longitude'})
)

# Watershed area: pull from shapefile-derived CSV
area = watersheds[['Unique_ID', 'area (sq mi)']].rename(columns={'area (sq mi)': 'watershed_area'})
area['Unique_ID'] = area['Unique_ID'].str.strip()

# Merge everything together
result = pivot.merge(coords, on='Unique_ID', how='left')
result = result.merge(area, on='Unique_ID', how='left')
result = result.merge(cw, on='Unique_ID', how='left')
result['cw_ind'] = result['cw_ind'].fillna(0).astype(int)

# Reorder columns: unique_id, latitude, longitude, watershed_area, cw_ind, then remaining species
col_order = ['Unique_ID', 'latitude', 'longitude', 'watershed_area', 'cw_ind'] + species_cols
result = result[col_order]

result.to_csv('./data/site_species_presence.csv', index=False)
print(f"Saved {len(result)} sites x {len(species_cols)} species to site_species_presence.csv")
result.head()


Saved 577 sites x 51 species to site_species_presence.csv


,Unique_ID,latitude,longitude,watershed_area,cw_ind,ABL,AE,ATS,BBH,BC,...,RT,SD,SL,SMB,STK,STS,TD,WP,YBH,YP
0,00-MIP_5_27_2021,43.001751,-71.661510,41.388147,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,00-MIP_8_22_2024,43.001751,-71.661510,41.388147,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,00B-PDB_9_14_2023,43.867904,-71.398396,8.083094,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,00C-BRK_7_13_2004,44.305489,-71.756152,5.624183,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,00C-BRK_8_2_2022,44.305511,-71.756144,5.624183,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
